# Dashboard del horneado de pan — versión final

Ecuación de calor 1D cilíndrica (Crank–Nicolson) con propiedades variables
miga/corteza, evaporación por *operator-splitting* y coeficiente de superficie
**`H_eff` = convección + radiación linealizada** 
Ejecuta las celdas en orden (Cell ▸ Run All). Mueve los sliders, pulsa
**Ejecutar simulación** y usa **Instante** para rebobinar el horneado.

**Instalación:** De ipywidgets (solo la primera vez).

In [5]:
!pip install ipywidgets


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


**Constantes físicas y propiedades del material**

In [6]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve_banded

Tf    = 373.16     # temperatura de evaporacion [K] (100 C)
dTb   = 0.5        # semiancho de la banda de cambio de fase [K]
Lv    = 2.3e6      # calor latente de evaporacion [J/kg]
rho_s = 241.76     # densidad del solido seco [kg/m3]
sigma = 5.67e-8    # constante de Stefan-Boltzmann [W/m2 K4]

def C2K(Tc): return Tc + 273.16
def K2C(Tk): return Tk - 273.16

def smoothH(T):
    """Heaviside suavizada centrada en Tf (~0 miga, ~1 corteza)."""
    return 1.0 / (1.0 + np.exp(-(T - Tf) / (dTb / 3.0)))

def k_eff(T):
    """Conductividad termica efectiva [W/m K], T en K."""
    k_miga = 0.9 / (1.0 + np.exp(-0.1 * (T - 353.16))) + 0.2
    s = smoothH(T)
    return (1.0 - s) * k_miga + s * 0.2

def Cp_sens(T, W):
    """Cp sensible (sin latente) [J/kg K], T en K."""
    Cps = 5.0 * T + 25.0
    Cpw = (5.207 - 73.17e-4 * T + 1.35e-5 * T**2) * 1000.0
    return Cps + W * Cpw

def rho_eff(T):
    """Densidad efectiva [kg/m3]. Purlis Ec.12: miga liviana, corteza densa."""
    s = smoothH(T)
    return (1.0 - s) * 180.61 + s * 321.31

**Solver paramétrico**

In [7]:
def simular_param(T_horno=180.0, H=15.0, R=0.04, t_max=1800.0, W_ini=0.60,
                  T_ini=25.0, Nx=140, dt=0.5, n_sub=3, n_frames=80, eps=0.9):
    dr = R/(Nx-1); rg = np.linspace(0, R, Nx); Nt = int(t_max/dt); dt_sub = dt/n_sub
    Tinf = C2K(T_horno)

    def construc(T, W):
        N=len(T); k=k_eff(T); cp=Cp_sens(T,W); rh=rho_eff(T)
        A=np.zeros((3,N)); B=np.zeros(N)
        for i in range(1,N-1):
            ri=rg[i]; r_p=0.5*(rg[i]+rg[i+1]); r_m=0.5*(rg[i]+rg[i-1])
            k_p=0.5*(k[i]+k[i+1]); k_m=0.5*(k[i]+k[i-1])
            coef=dt_sub/(rh[i]*cp[i]*ri*dr*dr); cp_=0.5*coef*r_p*k_p; cm_=0.5*coef*r_m*k_m
            A[2,i-1]=-cm_; A[1,i]=1+cp_+cm_; A[0,i+1]=-cp_
            B[i]=cm_*T[i-1]+(1-cp_-cm_)*T[i]+cp_*T[i+1]
        c0=2.0*dt_sub*k[0]/(rh[0]*cp[0]*dr*dr); A[1,0]=1+c0; A[0,1]=-c0
        B[0]=(1-c0)*T[0]+c0*T[1]
        # Superficie r=R con H_eff = conveccion + radiacion linealizada
        ks=k[-1]; Ts=T[-1]
        h_rad = eps*sigma*(Ts**2 + Tinf**2)*(Ts + Tinf)
        H_eff = H + h_rad
        gamma=H_eff*dr/ks; rN=0.5*dt_sub*ks/(rh[-1]*cp[-1]*dr*dr)
        A[2,-2]=-2*rN; A[1,-1]=1+2*rN*(1+gamma)
        B[-1]=2*rN*T[-2]+(1-2*rN*(1+gamma))*T[-1]+4*rN*gamma*Tinf
        return A, B

    def evap(T, W):
        for i in range(len(T)):
            if T[i] > Tf and W[i] > 0.0:
                rh=rho_eff(T[i]); cp=Cp_sens(T[i],W[i])
                Ex=rh*cp*(T[i]-Tf)*dr; Em=W[i]*rho_s*Lv*dr
                if Ex<=Em:
                    W[i]-=Ex/(rho_s*Lv*dr); T[i]=Tf
                else:
                    W[i]=0.0; T[i]=Tf+(Ex-Em)/(rh*cp*dr)
        return T, W

    T=np.full(Nx,C2K(T_ini)); W=np.full(Nx,W_ini)
    w_cell=rg.copy(); w_cell[0]=dr/8.0
    m_water0=np.sum(W*w_cell); m_solid=np.sum(1.0*w_cell)
    save=max(1,Nt//n_frames)
    tmin=[0.0]; Tc=[T_ini]; Ts=[T_ini]; WL=[0.0]; Hrad=[0.0]
    Tfield=[K2C(T.copy())]; Wfield=[W.copy()]
    for n in range(Nt):
        for _ in range(n_sub):
            A,B=construc(T,W); T=solve_banded((1,1),A,B); T,W=evap(T,W)
        if (n+1)%save==0:
            mw=np.sum(W*w_cell); tmin.append((n+1)*dt/60)
            Tc.append(K2C(T[0])); Ts.append(K2C(T[-1]))
            WL.append(100*(m_water0-mw)/(m_water0+m_solid))
            Tfield.append(K2C(T.copy())); Wfield.append(W.copy())
            Hrad.append(eps*sigma*(T[-1]**2+Tinf**2)*(T[-1]+Tinf))
    return dict(rg=rg, tmin=np.array(tmin), Tc=np.array(Tc), Ts=np.array(Ts),
                WL=np.array(WL), Hrad=np.array(Hrad),
                Tfield=np.array(Tfield), Wfield=np.array(Wfield),
                params=dict(T_horno=T_horno,H=H,R=R,t_max=t_max,W_ini=W_ini,eps=eps))

**Dashboard :** Al ejecutar esta celda y aparece el panel.

In [8]:
import ipywidgets as wd
from IPython.display import display, clear_output

plt.close('all')   # limpia figuras de corridas previas

sty = {'description_width':'140px', 'handle_color': 'pink'}; lay = wd.Layout(width='340px')
w_Th = wd.FloatSlider(value=200, min=120, max=260, step=5,  description='T_horno [C]', style=sty, layout=lay)
w_H  = wd.FloatSlider(value=15,  min=5,   max=40,  step=1,  description='H conv [W/m2K]', style=sty, layout=lay)
w_R  = wd.FloatSlider(value=3.0, min=1.0, max=8.0, step=0.5,description='R [cm]',      style=sty, layout=lay)
w_t  = wd.IntSlider(  value=30,  min=10,  max=90,  step=5,  description='t_max [min]', style=sty, layout=lay)
w_Wi = wd.FloatSlider(value=0.65,min=0.30,max=0.90,step=0.05,description='W_ini',     style=sty, layout=lay)
w_Ti = wd.IntSlider(  value=25,  min=4,   max=30,  step=1,  description='T_ini [C]',   style=sty, layout=lay)
w_Nx = wd.Dropdown(options=[60,100,140,200], value=140, description='Nx',    style=sty, layout=lay)
w_ns = wd.Dropdown(options=[1,2,3,4],         value=3,   description='n_sub', style=sty, layout=lay)

btn  = wd.Button(description='Ejecutar simulacion', icon='play',
                 layout=wd.Layout(width='340px'))
btn.style.button_color = '#e83375'
w_fr = wd.IntSlider(value=0, min=0, max=0, step=1, description='Instante',
                    continuous_update=False, style=sty, layout=wd.Layout(width='690px'))
out  = wd.Output()
state = {}

def dibujar(f):
    res = state.get('res')
    if res is None: return
    rg=res['rg']; R=res['params']['R']; W0=res['params']['W_ini']
    Ts_, Tc_ = res['Ts'], res['Tc']; tmin=res['tmin']; WL=res['WL']; r_cm=rg*100
    vmax=max(Ts_.max(), res['params']['T_horno'])
    ng=150; xx=np.linspace(-R,R,ng); XX,YY=np.meshgrid(xx,xx)
    RR=np.sqrt(XX**2+YY**2); mask=RR<=R
    img=np.where(mask, np.interp(RR,rg,res['Tfield'][f]), np.nan)
    with out:
        clear_output(wait=True)
        fig=plt.figure(figsize=(12,7))
        gs=fig.add_gridspec(2,3,height_ratios=[1.2,1],wspace=0.5,hspace=0.45)
        axD=fig.add_subplot(gs[0,0]); axT=fig.add_subplot(gs[0,1]); axW=fig.add_subplot(gs[0,2])
        ax1=fig.add_subplot(gs[1,0]); ax2=fig.add_subplot(gs[1,1]); axM=fig.add_subplot(gs[1,2]); axM.axis('off')
        im=axD.imshow(img,origin='lower',extent=[-R*100,R*100,-R*100,R*100],
                      cmap='inferno',vmin=25,vmax=vmax)
        axD.add_patch(plt.Circle((0,0),R*100,fill=False,color='white',lw=1.2))
        axD.set_title('Corte transversal'); axD.set_xlabel('x [cm]'); axD.set_ylabel('y [cm]')
        fig.colorbar(im,ax=axD,fraction=0.046,pad=0.04,label='T [C]')
        axT.plot(r_cm,res['Tfield'][f],color='orangered',lw=2.2)
        axT.axhline(100,color='gray',ls='--',lw=1)
        axT.fill_between(r_cm,0,res['Tfield'][f],color='orange',alpha=0.12)
        axT.set_ylim(0,vmax*1.05); axT.set_xlabel('r [cm]'); axT.set_ylabel('T [C]')
        axT.set_title('Perfil de temperatura'); axT.grid(alpha=0.3)
        axW.plot(r_cm,res['Wfield'][f],color='royalblue',lw=2.2)
        axW.set_ylim(-0.03,W0*1.1); axW.set_xlabel('r [cm]'); axW.set_ylabel('W [kg/kg]')
        axW.set_title('Perfil de humedad'); axW.grid(alpha=0.3)
        ax1.plot(tmin,Tc_,'gold',lw=2.2,label='Centro'); ax1.plot(tmin,Ts_,'coral',lw=2.2,label='Superficie')
        ax1.axvline(tmin[f],color='k',ls=':',lw=1); ax1.axhline(100,color='gray',ls='--',lw=1)
        ax1.set_xlabel('t [min]'); ax1.set_ylabel('T [C]'); ax1.legend(fontsize=8); ax1.grid(alpha=0.3)
        ax1.set_title('Temperatura vs tiempo')
        ax2.plot(tmin,WL,'magenta',lw=2.2); ax2.axvline(tmin[f],color='k',ls=':',lw=1)
        ax2.set_xlabel('t [min]'); ax2.set_ylabel('Perdida [%]'); ax2.grid(alpha=0.3)
        ax2.set_title('Perdida de peso vs tiempo')
        txt=(f"t = {tmin[f]:.1f} min\n\n"
             f"T centro      = {Tc_[f]:.1f} C\n"
             f"T superficie  = {Ts_[f]:.1f} C\n"
             f"Perdida peso  = {WL[f]:.1f} %\n"
             f"h_rad (sup.)  = {res['Hrad'][f]:.1f} W/m2K\n\n"
             f"Final:\n  T centro = {Tc_[-1]:.1f} C\n  perdida = {WL[-1]:.1f} %")
        axM.text(0,0.95,txt,va='top',ha='left',fontsize=11,family='monospace')
        display(fig)        # mostrar en el widget...
        plt.close(fig)      # ...y cerrar para que NO se vuelque de nuevo afuera

def _on_frame(ch):
    dibujar(ch['new'])

def on_run(b):
    btn.description='Calculando...'; btn.disabled=True
    state['res']=simular_param(T_horno=w_Th.value, H=w_H.value, R=w_R.value/100,
                               t_max=w_t.value*60, W_ini=w_Wi.value, T_ini=float(w_Ti.value),
                               Nx=w_Nx.value, n_sub=w_ns.value, n_frames=80)
    # actualizar el slider SIN disparar el observador (evita el doble dibujo)
    w_fr.unobserve(_on_frame, names='value')
    w_fr.max=len(state['res']['tmin'])-1
    w_fr.value=w_fr.max
    w_fr.observe(_on_frame, names='value')
    dibujar(w_fr.value)
    btn.description='Ejecutar simulacion'; btn.disabled=False

btn.on_click(on_run)
w_fr.observe(_on_frame, names='value')

panel = wd.VBox([
    wd.HTML('<b>Parametros de la ecuacion</b>'),
    wd.HBox([wd.VBox([w_Th,w_H,w_R,w_t]), wd.VBox([w_Wi,w_Ti,w_Nx,w_ns])]),
    btn, w_fr
])
display(panel, out)
on_run(None)   # primera corrida automatica

Output()